Ноутбук для использования в гугл колаб

In [2]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = ""

# Load the latest version
station = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "benhamner/sf-bay-area-bike-share",
  "station.csv",
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First station 5 records:", station.head())

trip = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "benhamner/sf-bay-area-bike-share",
  "trip.csv",
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First trip 5 records:", trip.head())

/tmp/ipykernel_3767/1603489223.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  station = kagglehub.load_dataset(


First station 5 records:    id                               name        lat        long  dock_count  \
0   2  San Jose Diridon Caltrain Station  37.329732 -121.901782          27   
1   3              San Jose Civic Center  37.330698 -121.888979          15   
2   4             Santa Clara at Almaden  37.333988 -121.894902          11   
3   5                   Adobe on Almaden  37.331415 -121.893200          19   
4   6                   San Pedro Square  37.336721 -121.894074          15   

       city installation_date  
0  San Jose          8/6/2013  
1  San Jose          8/5/2013  
2  San Jose          8/6/2013  
3  San Jose          8/5/2013  
4  San Jose          8/7/2013  


/tmp/ipykernel_3767/1603489223.py:22: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  trip = kagglehub.load_dataset(


Using Colab cache for faster access to the 'sf-bay-area-bike-share' dataset.
First trip 5 records:      id  duration       start_date        start_station_name  \
0  4576        63  8/29/2013 14:13  South Van Ness at Market   
1  4607        70  8/29/2013 14:42        San Jose City Hall   
2  4130        71  8/29/2013 10:16   Mountain View City Hall   
3  4251        77  8/29/2013 11:29        San Jose City Hall   
4  4299        83  8/29/2013 12:02  South Van Ness at Market   

   start_station_id         end_date          end_station_name  \
0                66  8/29/2013 14:14  South Van Ness at Market   
1                10  8/29/2013 14:43        San Jose City Hall   
2                27  8/29/2013 10:17   Mountain View City Hall   
3                10  8/29/2013 11:30        San Jose City Hall   
4                66  8/29/2013 12:04            Market at 10th   

   end_station_id  bike_id subscription_type zip_code  
0              66      520        Subscriber    94127  
1      

In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = SparkSession.builder.appName("Lab1").getOrCreate()


trips = spark.createDataFrame(trip)
stations = spark.createDataFrame(station)

trips = trips.filter(F.col("duration").isNotNull())

print("Количество поездок:", trips.count())
print("Количество станций:", stations.count())
trips.printSchema()
stations.printSchema()

Количество поездок: 669959
Количество станций: 70
root
 |-- id: long (nullable = true)
 |-- duration: long (nullable = true)
 |-- start_date: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: long (nullable = true)
 |-- end_date: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: long (nullable = true)
 |-- bike_id: long (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: long (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: string (nullable = true)



### Найти велосипед с максимальным временем пробега.

In [4]:
bike_total = trips.groupBy("bike_id") \
    .agg(F.sum("duration").alias("total_duration")) \
    .orderBy(F.desc("total_duration"))

max_bike = bike_total.first()
print(f"Велосипед c max duration: bike_id={max_bike['bike_id']}, "
      f"общее время={max_bike['total_duration']} сек ({max_bike['total_duration']/3600:.1f} ч)")
bike_total.show(10)

Велосипед c max duration: bike_id=535, общее время=18611693 сек (5169.9 ч)
+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    535|      18611693|
|    466|       3933272|
|    613|       2409014|
|    526|       2253019|
|    415|       2248886|
|    572|       2234149|
|    524|       2214314|
|    542|       2213422|
|    465|       2185170|
|    376|       2178177|
+-------+--------------+
only showing top 10 rows


### Найти наибольшее геодезическое расстояние между станциями.

In [7]:
from geopy.distance import geodesic


In [13]:
def distance_km(lat1, lon1, lat2, lon2):
    return geodesic((lat1, lon1), (lat2, lon2)).km

distance_udf = F.udf(distance_km, DoubleType())

stations1 = stations.alias("s1")
stations2 = stations.alias("s2")

pairs = stations1.crossJoin(stations2).filter(F.col("s1.id") < F.col("s2.id"))

distances = pairs.withColumn(
    "distance_km",
    distance_udf(F.col("s1.lat"), F.col("s1.long"), F.col("s2.lat"), F.col("s2.long"))
)

max_distance = distances.select(
    F.col("s1.id").alias("id1"),
    F.col("s1.name").alias("name1"),
    F.col("s2.id").alias("id2"),
    F.col("s2.name").alias("name2"),
    F.col("distance_km")
).orderBy(F.desc("distance_km")).first()

print(f"Максимальное расстояние: {max_distance.distance_km:.2f} км")
print(f"Станция 1: {max_distance.name1} (id={max_distance.id1})")
print(f"Станция 2: {max_distance.name2} (id={max_distance.id2})")

Максимальное расстояние: 69.92 км
Станция 1: SJSU - San Salvador at 9th (id=16)
Станция 2: Embarcadero at Sansome (id=60)


### Найти путь велосипеда с максимальным временем пробега через станции.

In [9]:
max_bike_id = max_bike["bike_id"]
print(f"Велосипед макс пробег: {max_bike_id}")

bike_path = trips.filter(F.col("bike_id") == max_bike_id) \
    .withColumn("start_ts", F.to_timestamp("start_date", "M/d/yyyy H:mm")) \
    .orderBy("start_ts") \
    .select("start_station_name", "end_station_name", "start_date", "end_date", "duration")

print(f"Количество поездок: {bike_path.count()}")
print("Путь через станции:")
bike_path.show(20, truncate=False)

Велосипед макс пробег: 535
Количество поездок: 1328

Путь через станции:
+---------------------------------------------+----------------------------------------+---------------+---------------+--------+
|start_station_name                           |end_station_name                        |start_date     |end_date       |duration|
+---------------------------------------------+----------------------------------------+---------------+---------------+--------+
|Post at Kearney                              |San Francisco Caltrain (Townsend at 4th)|8/29/2013 19:32|8/29/2013 19:53|1245    |
|San Francisco Caltrain (Townsend at 4th)     |San Francisco Caltrain 2 (330 Townsend) |8/29/2013 21:38|8/29/2013 21:45|423     |
|San Francisco Caltrain 2 (330 Townsend)      |Market at Sansome                       |8/30/2013 8:40 |8/30/2013 8:54 |842     |
|Market at Sansome                            |2nd at South Park                       |8/30/2013 9:10 |8/30/2013 9:19 |498     |
|2nd at Townsend 

### Найти количество велосипедов в системе.

In [10]:
bike_count = trips.select("bike_id").distinct().count()
print(f"колво велосипедов в системе: {bike_count}")

колво велосипедов в системе: 700


### Найти пользователей потративших на поездки более 3 часов.

In [12]:
total_hours = trips.groupBy("zip_code") \
    .agg((F.sum("duration") / 3600).alias("total_hours")) \
    .filter(F.col("total_hours") > 3) \
    .orderBy(F.desc("total_hours"))

print("Пользователи со временем поездок более 3 часов:")
total_hours.show(20, truncate=False)

Пользователи со временем поездок более 3 часов:
+--------+------------------+
|zip_code|total_hours       |
+--------+------------------+
|94107   |13821.433888888889|
|nil     |12701.541666666666|
|NaN     |7700.909166666666 |
|94105   |7110.035555555555 |
|94133   |6010.465277777777 |
|94102   |5313.339166666667 |
|94103   |5313.163333333333 |
|95531   |4797.333333333333 |
|94111   |3956.9436111111113|
|95112   |3539.5472222222224|
|94109   |3349.202222222222 |
|94040   |2168.8683333333333|
|94110   |2061.648888888889 |
|94117   |1917.0313888888888|
|94301   |1830.6605555555554|
|94041   |1743.4122222222222|
|94158   |1735.6019444444444|
|94306   |1541.8452777777777|
|94025   |1438.3991666666666|
|94108   |1424.3227777777777|
+--------+------------------+
only showing top 20 rows
